# فاز دوم — نوت‌بوک ارائه (Demo + Report)

این نوت‌بوک خروجی‌های فاز دوم را از `results/` می‌خواند و یک جمع‌بندی قابل ارائه تولید می‌کند.


In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

RESULTS = Path('results')
TABLES = RESULTS / 'tables'
FIGS = RESULTS / 'figures'

paths = {
  'model_comparison': TABLES / 'phase2_model_comparison.csv',
  'model_reports': TABLES / 'phase2_model_reports.json',
  'tuning_results': TABLES / 'phase2_tuning_results.csv',
  'best_model_report': TABLES / 'phase2_best_model_report.json',
  'threshold_sweep': TABLES / 'phase2_threshold_sweep_valid.csv',
  'threshold_summary': TABLES / 'phase2_threshold_summary.json',
}
for k,v in paths.items():
    print(f"{k:>16} -> {v} | exists={v.exists()}")


## 1) مقایسه مدل‌ها (Baseline Comparison)


In [ ]:
df_cmp = pd.read_csv(paths['model_comparison'])
df_cmp.sort_values(by=['valid_f1','valid_auc'], ascending=False)


## 2) نتایج دقیق‌تر هر مدل (valid/test)


In [ ]:
rep = json.loads(paths['model_reports'].read_text(encoding='utf-8'))
rows = []
for m, rr in rep['reports'].items():
    rows.append({
        'model': m,
        'valid_f1': rr['valid']['f1'],
        'valid_auc': rr['valid']['roc_auc'],
        'test_f1': rr['test']['f1'],
        'test_auc': rr['test']['roc_auc'],
        'test_acc': rr['test']['accuracy'],
    })
pd.DataFrame(rows).sort_values(['valid_f1','valid_auc'], ascending=False)


## 3) Hyperparameter Tuning


In [ ]:
df_tune = pd.read_csv(paths['tuning_results'])
df_tune.sort_values('cv_best_score', ascending=False).head(10)


In [ ]:
best = json.loads(paths['best_model_report'].read_text(encoding='utf-8'))
best['best_model'], best['best_cv_score']


In [ ]:
best['best_params']


### عملکرد Best Model روی Valid/Test (thr=0.5)


In [ ]:
best['valid']


In [ ]:
best['test']


## 4) Threshold Tuning + Calibration


In [ ]:
thr = json.loads(paths['threshold_summary'].read_text(encoding='utf-8'))
thr['calibration'], thr['best_threshold_valid']


In [ ]:
thr['test_at_0.5']


In [ ]:
thr['test_at_best_threshold']


## 5) نمایش نمودارهای کلیدی (اگر موجود باشند)


In [ ]:
import matplotlib.pyplot as plt

def show_if_exists(p):
    p = Path(p)
    if p.exists():
        img = plt.imread(p)
        plt.figure(figsize=(6,4))
        plt.imshow(img)
        plt.axis('off')
        plt.title(p.name)
        plt.show()
    else:
        print('missing:', p)

show_if_exists(FIGS / 'phase2_f1_vs_threshold_valid.png')
show_if_exists(FIGS / 'phase2_pr_curve_valid.png')
show_if_exists(FIGS / 'phase2_calibration_curve_valid.png')
show_if_exists(FIGS / 'phase2_cm_test_threshold_0.5.png')


## 6) اجرای دمو

### Streamlit
```bash
pip install streamlit
streamlit run app/streamlit_app.py
```

### CLI
```bash
python -m src.inference.predict_csv --input_csv data/raw/bank_marketing.csv --output_csv results/predictions_raw.csv
```
